# Shor order finding in Cirq → OpenQASM 2.0 → Quokka

A compiled **N=15, a=2** educational example matching the supplied Qiskit notebook. Eight qubits: four counting and four work. The modular arithmetic is specialized, not general-purpose factoring. Run cells sequentially. Remote Quokka availability and gate support are not guaranteed.

## 1. Install and import
`cirq-core` supplies the circuit simulator and OpenQASM 2.0 serializer. `requests` sends the QASM to the Quokka endpoint used in the uploaded Quokka notebook.

In [ ]:
%pip install -q cirq-core numpy matplotlib requests

In [ ]:
import cirq
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from fractions import Fraction
from math import gcd
import requests

N, a, n_count = 15, 2, 4
count = list(cirq.LineQubit.range(n_count))
work = list(cirq.LineQubit.range(n_count, n_count + 4))
qubit_order = count + work
print("Cirq version:", cirq.__version__)

## 2. Modular multiplication
Work register uses little-endian binary: `work[0]` represents the least significant bit. Three controlled swaps implement a cyclic left rotation of the **bit values**: `work[0] → work[1] → work[2] → work[3] → work[0]`. On the nonzero four-bit residues this implements multiplication by 2 modulo 15. For powers `2**j`, repeat the rotation `2**j mod 4` times. Each Fredkin gate is decomposed into two CNOTs and one Toffoli for QASM compatibility.

In [ ]:
def fredkin_decomposed(control, x, y):
    # Controlled swap: CNOT(x,y), CCX(control,y,x), CNOT(x,y)
    return [cirq.CNOT(x, y), cirq.TOFFOLI(control, y, x), cirq.CNOT(x, y)]

def controlled_mul_2_power_mod15(control, work, power):
    ops = []
    for _ in range(power % 4):
        for x, y in [(work[3], work[2]), (work[2], work[1]), (work[1], work[0])]:
            ops.extend(fredkin_decomposed(control, x, y))
    return ops

## 3. Inverse QFT
The counting register accumulates phases `k/r` from controlled modular multiplication. The inverse QFT maps those phases into computational-basis outcomes. We implement controlled phase as Rz/CNOT gates to keep the exported instruction set simple. Global phase is irrelevant to measurement.

In [ ]:
def controlled_phase(control, target, theta):
    # CP(theta), up to a global phase.
    return [
        cirq.rz(theta / 2).on(control),
        cirq.rz(theta / 2).on(target),
        cirq.CNOT(control, target),
        cirq.rz(-theta / 2).on(target),
        cirq.CNOT(control, target),
    ]

def inverse_qft(qubits):
    n = len(qubits)
    ops = []
    for i in range(n // 2):
        ops.append(cirq.SWAP(qubits[i], qubits[n - 1 - i]))
    for j in reversed(range(n)):
        for k in reversed(range(j + 1, n)):
            ops.extend(controlled_phase(qubits[k], qubits[j], -np.pi / (2 ** (k - j))))
        ops.append(cirq.H(qubits[j]))
    return ops

## 4. Construct the circuit
Start the work register in `|1⟩`. Put the counting qubits in uniform superposition, apply controlled powers of modular multiplication, then IQFT and measure. Counting output `c[0]` is the least significant bit; the printed bit strings use conventional most-significant-bit-first ordering.

In [ ]:
def make_shor15_circuit():
    ops = [cirq.X(work[0])]
    ops += [cirq.H(q) for q in count]
    for j, control in enumerate(count):
        ops.extend(controlled_mul_2_power_mod15(control, work, 2 ** j))
    ops.extend(inverse_qft(count))
    ops.append(cirq.measure(*count, key="c"))
    return cirq.Circuit(ops)

circuit = make_shor15_circuit()
print(circuit)

## 5. Run locally and verify
Expected ideal values: 0, 4, 8, 12, each with probability approximately 1/4. The test fails if the Fourier or modular multiplication circuit is incorrect.

In [ ]:
shots = 4096
result = cirq.Simulator(seed=42).run(circuit, repetitions=shots)
measurements = result.measurements["c"]  # shape (shots, 4); columns c[0]..c[3]
values = [sum(int(bit) << j for j, bit in enumerate(row)) for row in measurements]
counts_local = Counter(format(v, "04b") for v in values)
print("Local Cirq counts:", dict(sorted(counts_local.items())))
assert set(counts_local).issubset({"0000", "0100", "1000", "1100"}), "Unexpected outcomes: verify gate order"
plt.bar(sorted(counts_local), [counts_local[k] for k in sorted(counts_local)])
plt.title("Cirq Shor order finding: N=15, a=2")
plt.xlabel("Counting register (MSB first)")
plt.ylabel("Counts")
plt.show()

## 6. Export OpenQASM 2.0
Cirq generates the QASM. Quokka examples may require omitting the `qelib1.inc` line; only remove it if the server accepts built-in gate names without that include. Save the full version as a reference. The circuit contains `x`, `h`, `cx`, `ccx`, `swap`, `rz` and measurement instructions.

In [ ]:
qasm_text = str(cirq.QasmOutput(circuit.all_operations(), qubits=qubit_order))
assert "OPENQASM 2.0;" in qasm_text
with open("shor15_cirq_full.qasm", "w", encoding="utf-8") as f:
    f.write(qasm_text)
print(qasm_text[:2400])

# The Quokka notebook removed the qelib1.inc declaration.
qasm_quokka = "\n".join(
    line for line in qasm_text.splitlines()
    if line.strip() != 'include "qelib1.inc";'
) + "\n"
with open("shor15_cirq_quokka.qasm", "w", encoding="utf-8") as f:
    f.write(qasm_quokka)

## 7. Send to Quokka
This is the endpoint and JSON schema from the supplied Quokka notebook: `{"script": QASM, "count": shots}`. A timeout and error checks prevent a silent failure. **Remote execution is optional**; leave `RUN_QUOKKA=False` unless you want to send the request. Depending on the service, unsupported gates or circuit size may require further transpilation. Never interpret a failed API request as a quantum result.

In [ ]:
QUOKKA_URL = "http://quokka1.quokkacomputing.com/qsim/qasm"
RUN_QUOKKA = False

def run_quokka(qasm_program, shots=1024):
    response = requests.post(
        QUOKKA_URL, json={"script": qasm_program, "count": shots}, timeout=90
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("error_code", 0) != 0 or payload.get("error") not in (None, "no error"):
        raise RuntimeError(f"Quokka rejected circuit: {payload}")
    if not isinstance(payload.get("result"), dict):
        raise ValueError(f"Unexpected Quokka response: {payload}")
    return payload

quokka_payload = None
if RUN_QUOKKA:
    quokka_payload = run_quokka(qasm_quokka, shots=1024)
    print(quokka_payload)
else:
    print("Remote execution skipped. Set RUN_QUOKKA=True to submit.")

## 8. Interpret Quokka results
The Quokka example returns a list of per-shot classical bit arrays under `result["c"]`. **Check the server bit order** against a simple known circuit first. The code below assumes each row is `[c0,c1,c2,c3]`, as the notebook’s two-bit example suggests; this ordering has not been independently verified for four bits.

In [ ]:
def quokka_counts(payload, key="c"):
    rows = payload["result"][key]
    counts = Counter()
    for row in rows:
        if len(row) != n_count or any(int(b) not in (0, 1) for b in row):
            raise ValueError(f"Unexpected measurement row: {row}")
        value = sum(int(bit) << j for j, bit in enumerate(row))
        counts[format(value, "04b")] += 1
    return counts

counts = quokka_counts(quokka_payload) if quokka_payload else counts_local
print("Counts used for analysis:", dict(sorted(counts.items())))

## 9. Classical order recovery and factor extraction
Measured phases satisfy `m/16 ≈ k/r`. Continued fractions propose denominators, but reduced denominators can be divisors of the actual order. Test their multiples and confirm the **smallest** positive order using modular exponentiation. For this teaching example, search up to `N` (a safe small bound, not a scalable recovery method).

In [ ]:
def recover_order(counts, a=2, N=15, Q=16):
    for bitstring, frequency in counts.most_common():
        m = int(bitstring, 2)
        if m == 0:
            continue
        frac = Fraction(m, Q).limit_denominator(N)
        denominator = frac.denominator
        for r in range(denominator, N + 1, denominator):
            if pow(a, r, N) == 1:
                # Reduce to minimal positive order.
                for d in range(1, r + 1):
                    if r % d == 0 and pow(a, d, N) == 1:
                        return d, bitstring, frac
    return None, None, None

def factors_from_order(a, r, N):
    if r is None or r % 2:
        return None
    x = pow(a, r // 2, N)
    if x in (1, N - 1):
        return None
    p, q = gcd(x - 1, N), gcd(x + 1, N)
    if 1 < p < N and 1 < q < N and p * q == N:
        return p, q
    return None

r, measured, fraction = recover_order(counts, a, N, 2 ** n_count)
print("Measurement:", measured, "Phase fraction:", fraction)
print("Recovered order:", r)
print("Factors:", factors_from_order(a, r, N))
assert r == 4
assert set(factors_from_order(a, r, N)) == {3, 5}

## 10. Suggested experiments
1. Change `n_count` to 5 or 6, recreate registers and circuit, and inspect peaks at multiples of `Q/4`. Update output bit formatting accordingly.
2. Test each modular multiplication power against `pow(2, 2**j, 15)`.
3. Compare local Cirq and Quokka counts statistically.
4. Try a noisy local simulator and investigate the effect on phase recovery.
5. Explain why the zero phase gives no direct information about `r`, and why `1/2` can suggest a denominator smaller than the true order.

**Caution:** The current notebook hardcodes a four-bit output formatter for the default `n_count=4`; generalizing the register size requires updating the format strings and result parser.